In [2]:
from hullwhite import GeneratePathsHWAndAssetEuler, P0T, PriceEuropeanOptionHWMC
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Cálculo de Vega mediante diferencias finitas

Vega viene definida como la derivada del precio de la opción respecto a la volatilidad del activo subyacente:

$$
\text{Vega} = \frac{\partial V}{\partial \sigma_S}
$$

Desde el punto de vista financiero, la Vega mide cómo cambia el valor de la opción ante variaciones en la volatilidad del activo. En general, para opciones europeas, se espera una Vega positiva, ya que una mayor volatilidad incrementa la dispersión de posibles valores futuros del activo y, por tanto, aumenta el valor de la opcionalidad.

En este trabajo, la Vega se aproxima mediante diferencias finitas centradas:

$$
\text{Vega} \approx \frac{V(\sigma_S+h)-V(\sigma_S-h)}{2h}
$$

donde $h$ es una perturbación pequeña aplicada a la volatilidad del activo.

Utilizamos *common random numbers* para reducir la varianza del estimador Monte Carlo. Los precios correspondientes a $\sigma_S+h$ y $\sigma_S-h$ los calculamos utilizando la misma realización de números aleatorios, ue permite aislar el efecto de la perturbación sobre la volatilidad.

In [8]:
def VegaFD_HW_MC(NoOfPaths, NoOfSteps, T, P0T, lambd, eta, S0, sigma_S, rho, K, h, option_type, seed):
    """
    Calcula la Vega por diferencias finitas centradas respecto a la volatilidad del activo sigma_S
    en el modelo Hull–White + activo, usando common random numbers.

    Parámetros
    ----------
    h : Perturbación aplicada a sigma_S.
    seed : Semilla para reutilizar los mismos números aleatorios
    """

    # Precio con sigma_S + h
    paths_up = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0,
        sigma_S=sigma_S + h,
        rho=rho,
        seed=seed
    )

    price_up, _ = PriceEuropeanOptionHWMC(
        paths=paths_up,
        K=K,
        T=T,
        option_type=option_type
    )

    # Precio con sigma_S - h
    paths_down = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0,
        sigma_S=sigma_S - h,
        rho=rho,
        seed=seed
    )

    price_down, _ = PriceEuropeanOptionHWMC(
        paths=paths_down,
        K=K,
        T=T,
        option_type=option_type)

    vega_fd = (price_up - price_down) / (2.0 * h)

    return vega_fd, price_up, price_down

In [9]:
# Parámetros del modelo
NoOfPaths = 20000
NoOfSteps = 250
T = 1.0

lambd = 0.5
eta = 0.01

# Parámetros del activo
S0 = 100.0
sigma_S = 0.20
rho = -0.3

# Parámetros de la opción
K = 100.0
option_type = "call"

h_sigma = 0.01

vega_fd, price_up_sigma, price_down_sigma = VegaFD_HW_MC(
    NoOfPaths=NoOfPaths,
    NoOfSteps=NoOfSteps,
    T=T,
    P0T=P0T,
    lambd=lambd,
    eta=eta,
    S0=S0,
    sigma_S=sigma_S,
    rho=rho,
    K=K,
    h=h_sigma,
    option_type=option_type,
    seed=123
)

print(f"Precio con sigma + h = {price_up_sigma:.6f}")
print(f"Precio con sigma - h = {price_down_sigma:.6f}")
print(f"Vega FD             = {vega_fd:.6f}")

Precio con sigma + h = 10.759136
Precio con sigma - h = 10.010104
Vega FD             = 37.451609


#### Resultado numérico 

El valor obtenido para la Vega es positivo. Desde el punto de vista financiero, indica que un aumento en la volatilidad del activo incrementa el valor de la opción.
Una Vega de aproximadamente $37.45$ implica que un aumento de un punto porcentual en la volatilidad del activo, es decir, una variación de $0.01$ en $\sigma_S$, produciría un incremento de $0.37$ unidades en el precio de la opción.


#### Definición de Vega en el modelo con tipos de interés estocásticos

Aunque el modelo considerado incorpora tipos de interés estocásticos mediante Hull–White, la Vega que acabamos de estudiar se define, como la sensibilidad del precio de la opción respecto a la volatilidad del activo subyacente $\sigma_S$:

$$
\text{Vega} = \frac{\partial V}{\partial \sigma_S}
$$

Se analiza cómo cambia el precio de la opción cuando varía la volatilidad del activo, manteniendo la estructura de tipos estocásticos del modelo. En consecuencia, se trata de una Vega calculada dentro del modelo Hull–White, pero asociada a la incertidumbre del activo y no a la volatilidad del short rate.

La sensibilidad respecto a la volatilidad del tipo corto $\eta$ podría estudiarse también como una sensibilidad adicional distinta de la Vega estándar.

#### Cálculo de Vega mediante el método Pathwise

En el modelo con tipos de interés estocásticos, el precio de una call europea viene dado por:

$$
V_0 = \mathbb{E}^{\mathbb{Q}}\left[e^{-\int_0^T r_t\,dt}(S_T-K)^+\right]
$$

Aplicando el enfoque pathwise, se deriva directamente el payoff descontado respecto a $\sigma_S$. 

$$
S_{t_{i+1}} = S_{t_i}\exp\left((r_{t_i}-\tfrac12\sigma_S^2)\Delta t + \sigma_S\sqrt{\Delta t}Z_i^S\right)
$$

Iterando esta expresión hasta vencimiento, se obtiene que $S_T$ puede escribirse como una exponencial en la que aparece $\sigma_S$. Derivando respecto a esta variable, resulta:

$$
\frac{\partial S_T}{\partial \sigma_S}
=
S_T\left(-\sigma_S T + \sqrt{\Delta t}\sum_{i=0}^{N-1} Z_i^S\right)
$$

Por tanto, para una call europea, la Vega pathwise queda dada por:

$$
\text{Vega}_{PW}
=
\mathbb{E}^{\mathbb{Q}}
\left[
e^{-\int_0^T r_t\,dt}
\mathbf{1}_{\{S_T>K\}}
S_T\left(-\sigma_S T + \sqrt{\Delta t}\sum_{i=0}^{N-1} Z_i^S\right)
\right]
$$


In [10]:
def GeneratePathsHWAndAssetEuler(NoOfPaths, NoOfSteps, T, P0T, lambd, eta, S0, sigma_S, rho, seed=None):
    """
    Simula trayectorias de:
    - r_t: bajo Hull–White
    - S_t: activo financiero bajo medida riesgo-neutral

    Devuelve
    --------
    dict con:
        "time": malla temporal
        "R": trayectorias del short rate
        "S": trayectorias del activo
        "ZS": shocks gaussianos del activo
    """

    if seed is not None:
        np.random.seed(seed)

    dt_diff = 1e-4

    f0T = lambda t: -(np.log(P0T(t + dt_diff)) - np.log(P0T(t - dt_diff))) / (2 * dt_diff)

    r0 = f0T(1e-5)

    theta = lambda t: (
        1.0 / lambd * (f0T(t + dt_diff) - f0T(t - dt_diff)) / (2.0 * dt_diff)
        + f0T(t)
        + eta * eta / (2.0 * lambd * lambd) * (1.0 - np.exp(-2.0 * lambd * t))
    )

    dt = T / float(NoOfSteps)

    time = np.zeros([NoOfSteps + 1])

    R = np.zeros([NoOfPaths, NoOfSteps + 1])
    S = np.zeros([NoOfPaths, NoOfSteps + 1])

    # Guardamos los shocks del activo
    ZS = np.zeros([NoOfPaths, NoOfSteps])

    R[:, 0] = r0
    S[:, 0] = S0

    for i in range(NoOfSteps):

        Z1 = np.random.normal(0.0, 1.0, NoOfPaths)
        Z2 = np.random.normal(0.0, 1.0, NoOfPaths)

        if NoOfPaths > 1:
            Z1 = (Z1 - np.mean(Z1)) / np.std(Z1)
            Z2 = (Z2 - np.mean(Z2)) / np.std(Z2)

        Z_r = Z1
        Z_S = rho * Z1 + np.sqrt(1.0 - rho**2) * Z2

        # Guardamos el shock del activo
        ZS[:, i] = Z_S

        # Short rate
        R[:, i + 1] = (
            R[:, i]
            + lambd * (theta(time[i]) - R[:, i]) * dt
            + eta * np.sqrt(dt) * Z_r
        )

        # Activo
        S[:, i + 1] = (
            S[:, i]
            * np.exp((R[:, i] - 0.5 * sigma_S**2) * dt + sigma_S * np.sqrt(dt) * Z_S)
        )

        time[i + 1] = time[i] + dt

    return {"time": time, "R": R, "S": S, "ZS": ZS}

In [11]:
def VegaPathwise_HW_MC(paths, sigma_S, K, T, option_type="call"):
    """
    Calcula la Vega mediante el método pathwise
    en el modelo conjunto Hull–White + activo.
    """

    R = paths["R"]
    S = paths["S"]
    ZS = paths["ZS"]

    NoOfSteps = R.shape[1] - 1
    dt = T / NoOfSteps

    integral_r = np.sum(R[:, :-1] * dt, axis=1)
    discount = np.exp(-integral_r)

    ST = S[:, -1]

    # Término derivado dS_T / d sigma_S
    z_sum_term = np.sqrt(dt) * np.sum(ZS, axis=1)
    dST_dsigma = ST * (-sigma_S * T + z_sum_term)

    option_type = option_type.lower()

    if option_type == "call":
        indicator = (ST > K).astype(float)
        pathwise_samples = discount * indicator * dST_dsigma

    elif option_type == "put":
        indicator = (ST < K).astype(float)
        pathwise_samples = -discount * indicator * dST_dsigma

    else:
        raise ValueError("option_type debe ser 'call' o 'put'.")

    vega_pw = np.mean(pathwise_samples)
    stderr = np.std(pathwise_samples, ddof=1) / np.sqrt(len(pathwise_samples))

    return vega_pw, stderr

In [12]:
# Simulación conjunta
paths_joint = GeneratePathsHWAndAssetEuler(
    NoOfPaths=20000,
    NoOfSteps=250,
    T=1.0,
    P0T=P0T,
    lambd=0.5,
    eta=0.01,
    S0=100.0,
    sigma_S=0.20,
    rho=-0.3,
    seed=123
)

# Vega pathwise
vega_pw, stderr_vega_pw = VegaPathwise_HW_MC(
    paths=paths_joint,
    sigma_S=0.20,
    K=100.0,
    T=1.0,
    option_type="call"
)

print(f"Vega Pathwise  = {vega_pw:.6f}")
print(f"Error estándar = {stderr_vega_pw:.6f}")

Vega Pathwise  = 37.445173
Error estándar = 0.536827


In [13]:
print(f"Vega FD        = {vega_fd:.6f}")
print(f"Vega Pathwise  = {vega_pw:.6f}")
print(f"Diferencia     = {abs(vega_fd - vega_pw):.6f}")

Vega FD        = 37.451609
Vega Pathwise  = 37.445173
Diferencia     = 0.006436


#### Conclusión

Ambas estimaciones son muy próximas, lo que indica una elevada consistencia entre el método de diferencias finitas y el método pathwise también en el cálculo de la Vega.

El enfoque pathwise no solo resulta eficaz para la Delta, sino también para otras sensibilidades de primer orden, siempre que la dependencia del payoff respecto al parámetro de interés pueda derivarse de forma explícita.

En este caso, la derivación de $S_T$ respecto a la volatilidad del activo dentro del esquema log-Euler permite construir un estimador Monte Carlo directo de la Vega.

La proximidad entre los resultados obtenidos por ambos métodos refuerza la validez de la implementación y nos dice que el método pathwise es una herramienta adecuada para el cálculo de sensibilidades de primer orden también en el modelo con tipos de interés estocásticos.